# PYNQ.remote Cleanup Interactive Test

This notebook exercises remote cleanup for MMIO, buffers, GPIO, constructor auto-cleanup, overlay-download cleanup, and shutdown/restart behavior.

Run it while watching the `pynq-remote` logs on the target. Several cells intentionally perform stale operations; those cells should raise or print `NOT_FOUND` errors. That is the expected success condition.

## Setup

You should see no target-side resource activity from this cell except whatever happens when imports discover the remote device in your local environment. The helper functions below print stale-handle errors when an operation is expected to fail.

In [ ]:
import gc
import json
import os
from pathlib import Path

import grpc
import numpy as np

os.environ.setdefault("PYNQ_REMOTE_DEVICES", "192.168.2.197")
REMOTE_IP = os.environ["PYNQ_REMOTE_DEVICES"].split(",")[0].strip()

def find_repo_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "tests" / "resizer.xsa").exists():
            return candidate
    raise FileNotFoundError("Could not find tests/resizer.xsa from this notebook cwd")

REPO_ROOT = find_repo_root()
OVERLAY_PATH = REPO_ROOT / "tests" / "resizer.xsa"
STATE_PATH = REPO_ROOT / "remote_cleanup_notebook_state.json"

import pynq
from pynq import GPIO, Overlay
from pynq.pl_server.device import Device
from pynq.pl_server.remote_device import RemoteDevice
from pynq.remote import buffer_pb2, gpio_pb2, mmio_pb2

def reset_pynq_device_probe():
    if hasattr(Device, "_active_device"):
        delattr(Device, "_active_device")
    if hasattr(Device, "_devices"):
        delattr(Device, "_devices")

def probe_remote_device():
    """Use the normal PYNQ import/probe path, where auto_cleanup defaults on."""
    os.environ["PYNQ_REMOTE_DEVICES"] = REMOTE_IP
    reset_pynq_device_probe()
    remote_devices = [d for d in Device.devices if isinstance(d, RemoteDevice)]
    if not remote_devices:
        raise RuntimeError("No remote PYNQ devices discovered")
    return remote_devices[0]

def handle_epoch(handle_id):
    return int(handle_id.split(":", 1)[0])

def expect_not_found(label, fn, detail):
    try:
        fn()
    except grpc.RpcError as exc:
        print(f"{label}: {exc.code().name} - {exc.details()}")
        assert exc.code() == grpc.StatusCode.NOT_FOUND
        assert detail in exc.details()
        return
    raise AssertionError(f"{label} unexpectedly succeeded")

def create_gpio(device):
    base_path = GPIO.get_gpio_base_path(device=device)
    npins = GPIO.get_gpio_npins(device=device)
    if not (base_path and npins):
        print("GPIO sysfs base not available; GPIO portions will be skipped.")
        return None, None, None
    gpio_pin = GPIO.get_gpio_pin(0, device=device)
    gpio_path = f"/sys/class/gpio/gpio{gpio_pin}"
    if device.exists_file(gpio_path).exists:
        print(f"GPIO {gpio_path} already exported; GPIO creation skipped.")
        return None, gpio_pin, gpio_path
    gpio = GPIO(gpio_pin, "in", device=device)
    gpio.read()
    return gpio, gpio_pin, gpio_path

def create_resources(device, overlay=None):
    overlay = overlay or Overlay(str(OVERLAY_PATH), device=device)
    base_addr = overlay.ip_dict["resize_accel_0"]["phys_addr"]
    mmio = device.mmap(base_addr, 0x1000)
    mmio.read(0)
    buf = device.allocate(shape=(16,), dtype=np.uint32, cacheable=1)
    buf[:] = np.arange(16, dtype=np.uint32)
    buf.flush()
    gpio, gpio_pin, gpio_path = create_gpio(device)
    return {
        "overlay": overlay,
        "base_addr": base_addr,
        "mmio": mmio,
        "mmio_id": mmio.mmio_id,
        "buffer": buf,
        "buffer_id": buf.buffer_id,
        "gpio": gpio,
        "gpio_id": getattr(gpio, "_gpio_id", None),
        "gpio_pin": gpio_pin,
        "gpio_path": gpio_path,
    }

def release_resources(resources):
    for key, method in [("gpio", "release"), ("buffer", "freebuffer"), ("mmio", "close")]:
        obj = resources.get(key)
        if obj is None:
            continue
        try:
            getattr(obj, method)()
        except Exception as exc:
            print(f"Ignoring cleanup error for {key}: {exc}")
    gc.collect()

def check_stale_ids(device, resources, expected_state="stale", check_gpio_unexport=True):
    if expected_state == "stale":
        mmio_detail = "MMIO Object stale"
        buffer_detail = "Buffer Object stale"
        gpio_detail = "GPIO Object stale"
    elif expected_state == "not_found":
        mmio_detail = "MMIO Object not found"
        buffer_detail = "Buffer Object not found"
        gpio_detail = "GPIO Object not found"
    else:
        raise ValueError(f"Unknown expected state: {expected_state}")
    expect_not_found(
        "stale MMIO id",
        lambda: device._stub["mmio"].read(mmio_pb2.ReadRequest(mmio_id=resources["mmio_id"], offset=0, length=4, word_order="little")),
        mmio_detail,
    )
    expect_not_found(
        "stale buffer id",
        lambda: device._stub["buffer"].physical_address(buffer_pb2.AddressRequest(buffer_id=resources["buffer_id"])),
        buffer_detail,
    )
    if resources.get("gpio_id") is not None:
        expect_not_found(
            "stale GPIO id",
            lambda: device._stub["gpio"].read(gpio_pb2.GpioReadRequest(gpio_id=resources["gpio_id"])),
            gpio_detail,
        )
        if check_gpio_unexport:
            print("GPIO exported after cleanup:", device.exists_file(resources["gpio_path"]).exists)

def check_new_epoch(old_resources, fresh_resources):
    print("old MMIO -> fresh MMIO:", old_resources["mmio_id"], "->", fresh_resources["mmio_id"])
    print("old buffer -> fresh buffer:", old_resources["buffer_id"], "->", fresh_resources["buffer_id"])
    assert handle_epoch(fresh_resources["mmio_id"]) > handle_epoch(old_resources["mmio_id"])
    assert handle_epoch(fresh_resources["buffer_id"]) > handle_epoch(old_resources["buffer_id"])
    if old_resources.get("gpio_id") and fresh_resources.get("gpio_id"):
        print("old GPIO -> fresh GPIO:", old_resources["gpio_id"], "->", fresh_resources["gpio_id"])
        assert handle_epoch(fresh_resources["gpio_id"]) > handle_epoch(old_resources["gpio_id"])

print("Remote IP:", REMOTE_IP)
print("Overlay:", OVERLAY_PATH)


## 1. Individual Release

This creates one MMIO, one buffer, and optionally one GPIO, then releases each object explicitly.

You should see target logs like:

- `Function: release_mmio, ... released=true`
- `Freebuffer Request Received: ...` followed by `Buffer Successfuly Freed: ...`
- `Function: unexport, gpio_id=...` if GPIO is available

The stale checks at the end should print `NOT_FOUND` errors with `Object not found` details. Those errors are expected and mean the handles are gone.

In [ ]:
device = probe_remote_device()
overlay = Overlay(str(OVERLAY_PATH), device=device)
resources = create_resources(device, overlay=overlay)
print("created:", resources["mmio_id"], resources["buffer_id"], resources.get("gpio_id"))
release_resources(resources)
check_stale_ids(device, resources, expected_state="not_found")
device.close()


## 2. Explicit Global Cleanup

This creates resources, calls `device.cleanup()`, then deliberately uses stale objects and IDs.

You should see a target log like `Cleanup Request Received: buffers_freed=1, mmios_freed=..., gpios_freed=...`.

The stale operation checks should print `NOT_FOUND` errors with `Object stale` details for MMIO, buffer, and GPIO. After that, fresh resources should be allocated with a larger epoch, proving they do not alias old handles.

In [ ]:
device = probe_remote_device()
overlay = Overlay(str(OVERLAY_PATH), device=device)
old = create_resources(device, overlay=overlay)
print("old:", old["mmio_id"], old["buffer_id"], old.get("gpio_id"))
device.cleanup()
expect_not_found("stale MMIO object", lambda: old["mmio"].read(0), "MMIO Object stale")
expect_not_found("stale buffer object", lambda: old["buffer"].physical_address, "Buffer Object stale")
if old.get("gpio") is not None:
    expect_not_found("stale GPIO object", lambda: old["gpio"].read(), "GPIO Object stale")
check_stale_ids(device, old)
fresh = create_resources(device, overlay=overlay)
check_new_epoch(old, fresh)
check_stale_ids(device, old, check_gpio_unexport=False)
release_resources(fresh)
device.close()


## 3. Constructor Cleanup Across Kernel Restart

Run the next cell to create stale target resources with `auto_cleanup=False`. It writes the old handles to `remote_cleanup_notebook_state.json`.

You should see resource creation logs, but no cleanup from this cell.

After the cell finishes, restart the Python kernel. Then rerun only the setup cell at the top of this notebook, and continue with the verification cell below.

In [ ]:
device = RemoteDevice(ip_addr=REMOTE_IP, auto_cleanup=False)
overlay = Overlay(str(OVERLAY_PATH), device=device)
old = create_resources(device, overlay=overlay)
STATE_PATH.write_text(json.dumps({
    "base_addr": old["base_addr"],
    "mmio_id": old["mmio_id"],
    "buffer_id": old["buffer_id"],
    "gpio_id": old["gpio_id"],
    "gpio_pin": old["gpio_pin"],
    "gpio_path": old["gpio_path"],
}))
print("saved stale handles:", STATE_PATH)
print("old:", old["mmio_id"], old["buffer_id"], old.get("gpio_id"))
print("Restart the kernel now, then rerun Setup and continue below.")


### Verify Constructor Cleanup After Restart

This cell uses the normal `import pynq` / `Device.devices` probe path to construct a fresh remote device with default `auto_cleanup=True`. That constructor should clean the stale target resources left before the restart.

You should see a target log like `Cleanup Request Received: buffers_freed=1, mmios_freed=..., gpios_freed=...`.

Then this cell deliberately checks the old handles. You should see `NOT_FOUND` errors with `Object stale` details for the old MMIO/buffer/GPIO IDs, followed by fresh handles with a larger epoch.

In [ ]:
old = json.loads(STATE_PATH.read_text())
device = probe_remote_device()
check_stale_ids(device, old)
fresh = {
    "mmio": device.mmap(old["base_addr"], 0x1000),
    "buffer": device.allocate(shape=(16,), dtype=np.uint32, cacheable=1),
    "gpio": None,
    "gpio_id": None,
    "gpio_pin": old.get("gpio_pin"),
    "gpio_path": old.get("gpio_path"),
}
fresh["mmio"].read(0)
fresh["mmio_id"] = fresh["mmio"].mmio_id
fresh["buffer_id"] = fresh["buffer"].buffer_id
fresh["buffer"].physical_address
if old.get("gpio_pin") is not None:
    fresh["gpio"] = GPIO(old["gpio_pin"], "in", device=device)
    fresh["gpio_id"] = fresh["gpio"]._gpio_id
    fresh["gpio"].read()
check_new_epoch(old, fresh)
check_stale_ids(device, old, check_gpio_unexport=False)
release_resources(fresh)
device.close()


## 4. Overlay Download Cleanup

This cell creates resources, then loads the same overlay again. Full overlay download should cleanup old remote resources before the new overlay's MMIO setup.

You should see a cleanup request during the second `Overlay(...)`, then the stale operation checks should print `NOT_FOUND` errors with `Object stale` details. Fresh resources should use a newer epoch.

In [ ]:
device = probe_remote_device()
overlay1 = Overlay(str(OVERLAY_PATH), device=device)
old = create_resources(device, overlay=overlay1)
print("old before second overlay:", old["mmio_id"], old["buffer_id"], old.get("gpio_id"))
overlay2 = Overlay(str(OVERLAY_PATH), device=device)
expect_not_found("stale MMIO object after overlay", lambda: old["mmio"].read(0), "MMIO Object stale")
expect_not_found("stale buffer object after overlay", lambda: old["buffer"].physical_address, "Buffer Object stale")
if old.get("gpio") is not None:
    expect_not_found("stale GPIO object after overlay", lambda: old["gpio"].read(), "GPIO Object stale")
check_stale_ids(device, old)
fresh = create_resources(device, overlay=overlay2)
check_new_epoch(old, fresh)
check_stale_ids(device, old, check_gpio_unexport=False)
release_resources(fresh)
device.close()


## 5. Atexit / Shutdown Cleanup

Run the next cell, then shut down or restart the kernel.

You should see resource creation logs first. When the kernel exits cleanly, you should see a final cleanup request from `RemoteDevice.close()`.

After the restart, rerun Setup and then the verification cell. The old handles should produce `NOT_FOUND` errors with `Object stale` details. If they do not, the kernel did not run the `atexit` cleanup path and the next constructor cleanup should be investigated.

In [ ]:
device = probe_remote_device()
overlay = Overlay(str(OVERLAY_PATH), device=device)
live = create_resources(device, overlay=overlay)
STATE_PATH.write_text(json.dumps({
    "base_addr": live["base_addr"],
    "mmio_id": live["mmio_id"],
    "buffer_id": live["buffer_id"],
    "gpio_id": live["gpio_id"],
    "gpio_pin": live["gpio_pin"],
    "gpio_path": live["gpio_path"],
}))
print("live handles for atexit:", live["mmio_id"], live["buffer_id"], live.get("gpio_id"))
print("Now restart or shut down the kernel, then rerun Setup and the verification cell below.")


### Verify Atexit Cleanup After Restart

This uses `auto_cleanup=False` so the verification itself does not clean stale resources before checking them.

You should see `NOT_FOUND` errors with `Object stale` details for the old handles. Those errors are expected.

In [ ]:
old = json.loads(STATE_PATH.read_text())
device = RemoteDevice(ip_addr=REMOTE_IP, auto_cleanup=False)
check_stale_ids(device, old)
device.cleanup()


## 6. Legacy Bug Reproduction Mode

This section is for old images or deliberately disabled cleanup. It is not a fixed-behavior test. It is written to show the bugs this branch is meant to prevent.

Some cells below should print `BUG REPRODUCED` when run against an old image. On a fixed image, the same cells should usually print `FIXED` or `NOT_FOUND`.

The section avoids relying on the new cleanup RPC until explicitly noted, because older target images may not implement it.

In [ ]:
def legacy_device(auto_cleanup=False):
    try:
        return RemoteDevice(ip_addr=REMOTE_IP, auto_cleanup=auto_cleanup)
    except TypeError:
        print("RemoteDevice(auto_cleanup=...) is not available; using legacy constructor")
        return RemoteDevice(ip_addr=REMOTE_IP)

def probe_old_mmio(device, mmio_id):
    try:
        value = device._stub["mmio"].read(mmio_pb2.ReadRequest(mmio_id=mmio_id, offset=0, length=4, word_order="little")).data
        print(f"BUG REPRODUCED: stale MMIO {mmio_id} still reads {value}")
    except grpc.RpcError as exc:
        print(f"FIXED: stale MMIO {mmio_id} failed with {exc.code().name}: {exc.details()}")

def probe_old_buffer(device, buffer_id):
    try:
        address = device._stub["buffer"].physical_address(buffer_pb2.AddressRequest(buffer_id=buffer_id)).address
        print(f"BUG REPRODUCED: stale buffer {buffer_id} still has physical address {address}")
    except grpc.RpcError as exc:
        print(f"FIXED: stale buffer {buffer_id} failed with {exc.code().name}: {exc.details()}")

def probe_old_gpio(device, gpio_id):
    if gpio_id is None:
        print("No old GPIO handle was saved; skipping GPIO stale probe")
        return
    try:
        value = device._stub["gpio"].read(gpio_pb2.GpioReadRequest(gpio_id=gpio_id)).value
        print(f"BUG REPRODUCED: stale GPIO {gpio_id} still reads {value}")
    except grpc.RpcError as exc:
        print(f"FIXED: stale GPIO {gpio_id} failed with {exc.code().name}: {exc.details()}")

def maybe_cleanup(device):
    try:
        response = device.cleanup()
        print("cleanup RPC succeeded:", response)
    except Exception as exc:
        print("cleanup RPC unavailable or failed on this image:", repr(exc))


### Bug Repro A: Kernel Restart Leaves Server Resources Orphaned

Run the next cell on the old image, then restart the Python kernel. It creates MMIO, buffer, and optionally GPIO resources without calling cleanup. Old images should leave those target-side resources alive after the host kernel disappears.

You should see target logs for `get_mmio`, `Allocate Request Received`, and optionally `get_gpio`. You should not see a cleanup request from this cell.

In [ ]:
device = legacy_device(auto_cleanup=False)
overlay = Overlay(str(OVERLAY_PATH), device=device)
old = create_resources(device, overlay=overlay)
STATE_PATH.write_text(json.dumps({
    "base_addr": old["base_addr"],
    "mmio_id": old["mmio_id"],
    "buffer_id": old["buffer_id"],
    "gpio_id": old["gpio_id"],
    "gpio_pin": old["gpio_pin"],
    "gpio_path": old["gpio_path"],
}))
print("Saved stale handles for legacy repro:", old["mmio_id"], old["buffer_id"], old.get("gpio_id"))
print("Restart the kernel now. Then rerun Setup and the next legacy probe cell.")


### Probe A After Restart

Rerun Setup first, then run this cell.

On an old image, you may see `BUG REPRODUCED` because stale server-side resources still exist after the host kernel restart.

On a fixed image, you should see `FIXED` / `NOT_FOUND` for stale handles if `atexit` or the next constructor cleanup removed the resources.

In [ ]:
old = json.loads(STATE_PATH.read_text())
device = legacy_device(auto_cleanup=False)
probe_old_mmio(device, old["mmio_id"])
probe_old_buffer(device, old["buffer_id"])
probe_old_gpio(device, old.get("gpio_id"))
maybe_cleanup(device)


### Bug Repro B: Repeated Overlay Loads Without Cleanup Leave Old Handles Live

This mimics the old behavior by using `auto_cleanup=False`. It creates resources, loads the overlay again, and then probes the old handles.

You should see old target images keep the old buffer/MMIO/GPIO handles alive. That is the no-overlay-cleanup bug. On the fixed default path, this is prevented by full-overlay-download cleanup when `auto_cleanup=True`.

In [ ]:
device = legacy_device(auto_cleanup=False)
overlay1 = Overlay(str(OVERLAY_PATH), device=device)
old = create_resources(device, overlay=overlay1)
print("old handles before second overlay:", old["mmio_id"], old["buffer_id"], old.get("gpio_id"))
overlay2 = Overlay(str(OVERLAY_PATH), device=device)
probe_old_mmio(device, old["mmio_id"])
probe_old_buffer(device, old["buffer_id"])
probe_old_gpio(device, old.get("gpio_id"))
maybe_cleanup(device)


### Bug Repro C: Stale Handle Aliasing If IDs Reset Without Epochs

This is for builds that have a cleanup-like reset but do not include epoch-prefixed handles. The bad behavior is: cleanup removes old resources, a fresh resource gets the same handle string/number, and the stale host object accidentally operates on the new server object.

You should see one of these outcomes:

- `FIXED`: old handle fails and fresh handle has a newer epoch, such as `0:0 -> 1:0`.
- `BUG REPRODUCED`: old and fresh handles are equal, or the old stale operation succeeds after fresh allocation.
- `cleanup RPC unavailable`: the image is too old for this particular aliasing repro.

In [ ]:
device = legacy_device(auto_cleanup=False)
overlay = Overlay(str(OVERLAY_PATH), device=device)
old = create_resources(device, overlay=overlay)
old_mmio_id = old["mmio_id"]
old_buffer_id = old["buffer_id"]
old_gpio_id = old.get("gpio_id")
print("old handles:", old_mmio_id, old_buffer_id, old_gpio_id)

try:
    device.cleanup()
except Exception as exc:
    print("cleanup RPC unavailable; cannot run aliasing repro on this image:", repr(exc))
else:
    fresh = create_resources(device, overlay=overlay)
    print("fresh handles:", fresh["mmio_id"], fresh["buffer_id"], fresh.get("gpio_id"))
    if fresh["mmio_id"] == old_mmio_id or fresh["buffer_id"] == old_buffer_id or (old_gpio_id and fresh.get("gpio_id") == old_gpio_id):
        print("BUG REPRODUCED: fresh resource reused an old handle identity")
    else:
        print("FIXED: fresh handles differ from old handles")
    probe_old_mmio(device, old_mmio_id)
    probe_old_buffer(device, old_buffer_id)
    probe_old_gpio(device, old_gpio_id)
    release_resources(fresh)
release_resources(old)
maybe_cleanup(device)
